<a href="https://www.kaggle.com/code/prantakumerpandit390/lab-dm?scriptVersionId=360396355" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## --> Question :

### Add the kaggle file link after completing the tasks:
### 1. Objective: What association rule mining is intended to discover.
### 2. Dataset: The transaction structure and number of transactions.
### 3. Preprocessing: How transactions were converted to Boolean values.
### 4. Frequent itemsets: Results using at least two support thresholds.
### 5. Association rules: At least three rules with support, confidence, and lift.
### 6. Discussion: Business interpretation and limitations.
### 7. Conclusion: What was learned from the experiment

---

## 1. Objective: What Association Rule Mining is Intended to Discover

Association Rule Mining (commonly known as Market Basket Analysis) is intended to discover hidden buying patterns, relationships, and co-occurrences between items in large transaction datasets. 

Its main goal is to find out which products customers frequently buy together in the same shopping trip, written in the form of an **"If -> Then"** rule (`If a customer buys Item A -> they are also likely to buy Item B`). Retailers use these discovered patterns to:
- Place related products near each other on store shelves.
- Create product bundles and promotional discounts.
- Recommend complementary items to customers at checkout.

---

## Dataset Mounting

In [1]:
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("prasad22/retail-transactions-dataset")
print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/prasad22/retail-transactions-dataset


---
## Task 2 , Dataset: The transaction structure and number of transactions.

In [2]:
import os
import pandas as pd

# Load the CSV file from the path downloaded above
csv_file = os.path.join(path, "Retail_Transactions_Dataset.csv")
df = pd.read_csv(csv_file)

print("Total Number of Transactions (Rows):", df.shape[0])
print("Total Number of Columns:", df.shape[1])
print("\nColumn Names:", list(df.columns))
print("\nFirst 5 Rows:")
print(df[['Transaction_ID', 'Product', 'Total_Items', 'Total_Cost']].head())

Total Number of Transactions (Rows): 1000000
Total Number of Columns: 13

Column Names: ['Transaction_ID', 'Date', 'Customer_Name', 'Product', 'Total_Items', 'Total_Cost', 'Payment_Method', 'City', 'Store_Type', 'Discount_Applied', 'Customer_Category', 'Season', 'Promotion']

First 5 Rows:
   Transaction_ID                                            Product  \
0      1000000000        ['Ketchup', 'Shaving Cream', 'Light Bulbs']   
1      1000000001  ['Ice Cream', 'Milk', 'Olive Oil', 'Bread', 'P...   
2      1000000002                                        ['Spinach']   
3      1000000003                             ['Tissues', 'Mustard']   
4      1000000004                                      ['Dish Soap']   

   Total_Items  Total_Cost  
0            3       71.65  
1            2       25.93  
2            6       41.49  
3            1       39.34  
4           10       16.42  


---
## Task 3. Preprocessing: How Transactions Were Converted to Boolean Values

Association rule mining algorithms (such as Apriori and FP-Growth) cannot directly process raw text arrays. They require a **One-Hot Encoded Boolean Matrix** where each row represents a transaction, each column represents a distinct retail product, and each cell contains a Boolean value (`True` if the product was purchased in that transaction, and `False` otherwise).

To achieve a clean, full-scale transformation across **all 1,000,000 transactions**, the following preprocessing pipeline was executed:
1. **Missing Value & Format Verification:** Checked the `Product` column for null entries (`0` missing values found).
2. **String-to-List Parsing (`ast.literal_eval`):** In the raw CSV file, the `Product` column is stored as a string representation of a list (e.g., `"['Ketchup', 'Shaving Cream']"`). We parsed each string into an actual Python list object.
3. **Intra-Transaction Deduplication & Whitespace Stripping:** Standard association rule mining evaluates the *presence* or *absence* of an item in a basket rather than duplicate occurrences. Each transaction list was converted into a Python `set()` and stripped of trailing spaces to eliminate duplicate product entries within the same basket.
4. **Vocabulary Extraction:** Identified all unique items across the 1,000,000 transactions, resulting in exactly **81 unique products**.
5. **Boolean Matrix Construction:** Constructed a $1,000,000 \times 81$ Boolean (`True`/`False`) DataFrame (`df_bool`). For each transaction $i$ and product $j$, the matrix entry is set to `True` if product $j$ is present in transaction $i$, and `False` otherwise. Because `bool` uses only 1 byte per cell, the entire 1,000,000-row matrix occupies only ~77.25 MB of memory.


---
## Task 4 : Frequent itemsets: Results using at least two support thresholds.


### Data Cleaning with boolean check

In [3]:
import ast
import numpy as np

print(" DATA CLEANING & INSPECTION :")
missing_count = df['Product'].isnull().sum()
print(f"Missing values in 'Product' column: {missing_count}")


raw_transactions = [ast.literal_eval(x) for x in df['Product']]
clean_transactions = [ sorted(list(set(item.strip() for item in basket))) for basket in raw_transactions ]

basket_sizes = [len(b) for b in clean_transactions]
single_item_baskets = sum(1 for size in basket_sizes if size == 1)
multi_item_baskets = sum(1 for size in basket_sizes if size >= 2)

print(f"Total Transactions Processed: {len(clean_transactions):,}")
print(f"Single-Item Baskets (1 unique item): {single_item_baskets:,} ({single_item_baskets/len(clean_transactions)*100:.2f}%)")
print(f"Multi-Item Baskets (2+ unique items): {multi_item_baskets:,} ({multi_item_baskets/len(clean_transactions)*100:.2f}%)")
print(f"Average Unique Items per Basket: {np.mean(basket_sizes):.2f}")

print("\n ONE-HOT BOOLEAN CONVERSION (ALL 1,000,000 ROWS) : ")

unique_products = sorted(list(set(item for basket in clean_transactions for item in basket)))
product_to_idx = {prod: idx for idx, prod in enumerate(unique_products)}
print(f"Total Unique Products Identified: {len(unique_products)}")


bool_matrix = np.zeros((len(clean_transactions), len(unique_products)), dtype=bool)

for row_idx, basket in enumerate(clean_transactions):
    for item in basket:
        bool_matrix[row_idx, product_to_idx[item]] = True

# Converting to Pandas DataFrame with Boolean 
df_bool = pd.DataFrame(bool_matrix, columns=unique_products)

print("Boolean Matrix Shape:", df_bool.shape)
print("Memory Usage of Boolean Matrix: {:.2f} MB".format(df_bool.memory_usage().sum() / (1024**2)))
print("\nFirst 5 Rows of Boolean DataFrame (First 8 Product Columns):")
print(df_bool.iloc[:5, :8])

 DATA CLEANING & INSPECTION :
Missing values in 'Product' column: 0
Total Transactions Processed: 1,000,000
Single-Item Baskets (1 unique item): 202,256 (20.23%)
Multi-Item Baskets (2+ unique items): 797,744 (79.77%)
Average Unique Items per Basket: 2.95

 ONE-HOT BOOLEAN CONVERSION (ALL 1,000,000 ROWS) : 
Total Unique Products Identified: 81
Boolean Matrix Shape: (1000000, 81)
Memory Usage of Boolean Matrix: 77.25 MB

First 5 Rows of Boolean DataFrame (First 8 Product Columns):
   Air Freshener  Apple  BBQ Sauce  Baby Wipes  Banana  Bath Towels   Beef  \
0          False  False      False       False   False        False  False   
1          False  False      False       False   False        False  False   
2          False  False      False       False   False        False  False   
3          False  False      False       False   False        False  False   
4          False  False      False       False   False        False  False   

   Bread  
0  False  
1   True  
2  False  
3  

## Main Task #4

In [4]:
import numpy as np
import pandas as pd
from itertools import combinations

# Converting boolean DataFrame to int32 matrix for ultra-fast matrix multiplication
X = df_bool.values.astype(np.int32)
n_transactions = X.shape[0]
products = np.array(df_bool.columns)


item_counts = X.sum(axis=0)
item_supports = item_counts / n_transactions

df_1_itemsets = pd.DataFrame({
    'itemsets': [f"({p})" for p in products],
    'length': 1,
    'count': item_counts,
    'support': item_supports
}).sort_values(by='support', ascending=False).reset_index(drop=True)

print("--- SUMMARY OF 1-ITEMSET SUPPORTS ---")
print(f"Highest single-item support: {df_1_itemsets['support'].max():.6f} ({df_1_itemsets.loc[0, 'itemsets']})")
print(f"Average single-item support: {df_1_itemsets['support'].mean():.6f}")
print(f"Lowest single-item support:  {df_1_itemsets['support'].min():.6f}")


# Using dot product X.T @ X computes all 3,240 product pairs across 1,000,000 rows in 1 second!

co_matrix = X.T @ X

pair_list = []
for i in range(len(products)):
    for j in range(i + 1, len(products)):
        count_ij = co_matrix[i, j]
        support_ij = count_ij / n_transactions
        pair_list.append({
            'itemsets': f"({products[i]}, {products[j]})",
            'item_A': products[i],
            'item_B': products[j],
            'length': 2,
            'count': count_ij,
            'support': support_ij
        })

df_2_itemsets = pd.DataFrame(pair_list).sort_values(by='support', ascending=False).reset_index(drop=True)

print("\n--- SUMMARY OF 2-ITEMSET (PAIR) SUPPORTS ---")
print(f"Total possible unique 2-item pairs (81 choose 2): {len(df_2_itemsets):,}")
print(f"Highest 2-item pair support: {df_2_itemsets['support'].max():.6f} ({df_2_itemsets.loc[0, 'itemsets']})")
print(f"Average 2-item pair support: {df_2_itemsets['support'].mean():.6f}")
print(f"Lowest 2-item pair support:  {df_2_itemsets['support'].min():.6f}")


df_all_itemsets = pd.concat([
    df_1_itemsets[['itemsets', 'length', 'count', 'support']],
    df_2_itemsets[['itemsets', 'length', 'count', 'support']]
], ignore_index=True)


# Threshold 1: High Support (min_support = 0.0370 or 3.70%)
thresh_1 = 0.0370
freq_t1 = df_all_itemsets[df_all_itemsets['support'] >= thresh_1].sort_values(by='support', ascending=False)

# Threshold 2: Medium Support (min_support = 0.0350 or 3.50%)
thresh_2 = 0.0350
freq_t2 = df_all_itemsets[df_all_itemsets['support'] >= thresh_2].sort_values(by='support', ascending=False)

# Threshold 3: Pair-Discovery Support (min_support = 0.0011 or 0.11%)
thresh_3 = 0.0011
freq_t3 = df_all_itemsets[df_all_itemsets['support'] >= thresh_3].sort_values(by='support', ascending=False)
freq_t3_pairs = df_2_itemsets[df_2_itemsets['support'] >= thresh_3]


print("\n TASK 4 RESULTS: COMPARISON ACROSS SUPPORT THRESHOLDS")

print(f"1. Threshold = {thresh_1} (3.70%): Found {len(freq_t1)} frequent itemsets (1-itemsets: {sum(freq_t1['length']==1)}, 2-itemsets: {sum(freq_t1['length']==2)})")
print(f"2. Threshold = {thresh_2} (3.50%): Found {len(freq_t2)} frequent itemsets (1-itemsets: {sum(freq_t2['length']==1)}, 2-itemsets: {sum(freq_t2['length']==2)})")
print(f"3. Threshold = {thresh_3} (0.11%): Found {len(freq_t3)} frequent itemsets (1-itemsets: {sum(freq_t3['length']==1)}, 2-itemsets: {sum(freq_t3['length']==2)})")

print("\nTop 10 Most Frequent Single Items (1-Itemsets):")
print(df_1_itemsets.head(10).to_string(index=False))

print("\nTop 10 Most Frequent Product Pairs (2-Itemsets):")
print(df_2_itemsets[['itemsets', 'length', 'count', 'support']].head(10).to_string(index=False))

--- SUMMARY OF 1-ITEMSET SUPPORTS ---
Highest single-item support: 0.070949 ((Toothpaste))
Average single-item support: 0.036430
Lowest single-item support:  0.035594

--- SUMMARY OF 2-ITEMSET (PAIR) SUPPORTS ---
Total possible unique 2-item pairs (81 choose 2): 3,240
Highest 2-item pair support: 0.002412 ((Tomatoes, Toothpaste))
Average 2-item pair support: 0.001185
Lowest 2-item pair support:  0.001042

 TASK 4 RESULTS: COMPARISON ACROSS SUPPORT THRESHOLDS
1. Threshold = 0.037 (3.70%): Found 1 frequent itemsets (1-itemsets: 1, 2-itemsets: 0)
2. Threshold = 0.035 (3.50%): Found 81 frequent itemsets (1-itemsets: 81, 2-itemsets: 0)
3. Threshold = 0.0011 (0.11%): Found 3193 frequent itemsets (1-itemsets: 81, 2-itemsets: 3112)

Top 10 Most Frequent Single Items (1-Itemsets):
       itemsets  length  count  support
   (Toothpaste)       1  70949 0.070949
    (Ice Cream)       1  36485 0.036485
         (Soap)       1  36438 0.036438
       (Orange)       1  36342 0.036342
          (Jam)  

### About task 4 , Frequent Itemsets: Results Using Multiple Support Thresholds

To understand how the minimum support threshold ($\text{min\_support}$) affects pattern discovery across all **1,000,000 transactions**, we evaluated three distinct thresholds based on the underlying probability distribution of the 81 products:

#### Comparison of Support Thresholds
| Support Threshold ($\text{min\_support}$) | Minimum Transaction Count | 1-Itemsets Found | 2-Itemsets (Pairs) Found | Total Frequent Itemsets | Analytical Observation |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Threshold 1: `0.0370` (3.70%)** | 37,000 | 1 | 0 | **1** | Only `(Toothpaste)` qualifies (`support = 0.070949`), as it appears nearly twice as frequently as any other product. |
| **Threshold 2: `0.0350` (3.50%)** | 35,000 | 81 | 0 | **81** | Captures all 81 individual products (whose supports range from `0.035594` to `0.070949`), but is still too high to capture multi-item combinations. |
| **Threshold 3: `0.0011` (0.11%)** | 1,100 | 81 | 3,112 | **3,193** | Captures all 81 single items plus **3,112 out of 3,240** possible 2-item pairs (the highest pair support is `0.002412` for `(Tomatoes, Toothpaste)`). |

#### Key Findings from Frequent Itemset Mining
1. **Dominance of Toothpaste (1-Itemsets):** While 80 out of the 81 products have a uniform support clustering tightly around **3.60%** (~36,000 transactions each), `(Toothpaste)` is a strong statistical outlier with a support of **7.0949%** (`70,949` transactions), followed by `(Ice Cream)` at **3.6485%** (`36,485` transactions) and `(Soap)` at **3.6438%** (`36,438` transactions).
2. **Co-occurrence Scale (2-Itemsets):** Because average basket size is small (**2.95 unique items** spread across **81 products**), the mathematical probability of any two specific items co-occurring is naturally low (averaging `0.001185` or `0.1185%`). Lowering the threshold from `0.0350` to `0.0011` is essential to transition from single-item popularity to multi-item co-occurrence pairs.
3. **Top Frequent Pairs:** The top 10 most frequent 2-itemsets all involve `Toothpaste` paired with popular items, led by `(Tomatoes, Toothpaste)` (`support = 0.002412`, `2,412` transactions), `(Deodorant, Toothpaste)` (`support = 0.002405`, `2,405` transactions), and `(Baby Wipes, Toothpaste)` (`support = 0.002396`, `2,396` transactions).

---
### TASK 5 PART A: Code for create table 1 & table 2 for 2-item rules

In [5]:
#  TASK 5: (Part A)
#EXACT ASSOCIATION RULES (SUPPORT, CONFIDENCE, LIFT

# Create lookup dictionary for single-item support: P(A)
supp_dict = dict(zip(products, item_supports))

rules_list = []
for idx, row in df_2_itemsets.iterrows():
    A = row['item_A']
    B = row['item_B']
    supp_AB = row['support']
    count_AB = row['count']
    
    supp_A = supp_dict[A]
    supp_B = supp_dict[B]
    
    # Lift equation
    lift = supp_AB / (supp_A * supp_B)
    
    # Rule 1: A -> B
    conf_A_to_B = supp_AB / supp_A
    rules_list.append({
        'Rule': f"{{{A}}} -> {{{B}}}",
        'Antecedent (A)': A,
        'Consequent (B)': B,
        'Antecedent_Support': supp_A,
        'Consequent_Support': supp_B,
        'Support': supp_AB,
        'Confidence': conf_A_to_B,
        'Lift': lift,
        'Pair_Count': count_AB
    })
    
    # Rule 2: B -> A
    conf_B_to_A = supp_AB / supp_B
    rules_list.append({
        'Rule': f"{{{B}}} -> {{{A}}}",
        'Antecedent (A)': B,
        'Consequent (B)': A,
        'Antecedent_Support': supp_B,
        'Consequent_Support': supp_A,
        'Support': supp_AB,
        'Confidence': conf_B_to_A,
        'Lift': lift,
        'Pair_Count': count_AB
    })

df_rules = pd.DataFrame(rules_list)

print(f"Total Association Rules Evaluated: {len(df_rules):,}")

print("\n TABLE 1: TOP 5 ASSOCIATION RULES BY HIGHEST CONFIDENCE ")
top_conf = df_rules.sort_values(by=['Confidence', 'Lift'], ascending=False).head(5)
print(top_conf[['Rule', 'Support', 'Confidence', 'Lift', 'Pair_Count']].to_string(index=False))

print("\n TABLE 2: TOP 5 ASSOCIATION RULES BY HIGHEST LIFT (STRONGEST CORRELATION) ")

# Now Since A->B and B->A share the same Lift, keep the direction with higher Confidence

top_lift = df_rules.sort_values(by=['Lift', 'Confidence'], ascending=False).drop_duplicates(subset=['Pair_Count', 'Support']).head(5)
print(top_lift[['Rule', 'Support', 'Confidence', 'Lift', 'Pair_Count']].to_string(index=False))

Total Association Rules Evaluated: 6,480

 TABLE 1: TOP 5 ASSOCIATION RULES BY HIGHEST CONFIDENCE 
                       Rule  Support  Confidence     Lift  Pair_Count
 {Tomatoes} -> {Toothpaste} 0.002412    0.066959 0.943764        2412
   {Onions} -> {Toothpaste} 0.002388    0.066363 0.935360        2388
   {Cheese} -> {Toothpaste} 0.002386    0.066324 0.934810        2386
{Deodorant} -> {Toothpaste} 0.002405    0.066294 0.934384        2405
    {Apple} -> {Toothpaste} 0.002368    0.066241 0.933649        2368

 TABLE 2: TOP 5 ASSOCIATION RULES BY HIGHEST LIFT (STRONGEST CORRELATION) 
                                      Rule  Support  Confidence     Lift  Pair_Count
                      {Butter} -> {Banana} 0.001295    0.036133 0.997483        1295
{Feminine Hygiene Products} -> {Deodorant} 0.001275    0.035387 0.975445        1275
                   {BBQ Sauce} -> {Yogurt} 0.001267    0.035150 0.972190        1267
                         {Razors} -> {Mop} 0.001251    0.034867 0


## About task 5 , Association Rules: Support, Confidence, and Lift

We generated association rules (`If Item A is bought -> Item B is bought`) using three metrics:
- **Support:** How often the items are bought together in the whole dataset.
- **Confidence:** The chance (%) that Item B is bought if Item A is already in the basket.
- **Lift:** Strength of the rule. `Lift > 1.0` means a positive connection (they help each other sell). `Lift < 1.0` means no positive connection.

#### 1. Two-Item Rules (`A -> B`)

**Table 1: Top Rules by Confidence**
| Rule | Support | Confidence | Lift | Count |
| :--- | :--- | :--- | :--- | :--- |
| `{Tomatoes} -> {Toothpaste}` | 0.002412 | 6.70% | 0.9438 | 2,412 |
| `{Onions} -> {Toothpaste}` | 0.002388 | 6.64% | 0.9354 | 2,388 |
| `{Cheese} -> {Toothpaste}` | 0.002386 | 6.63% | 0.9348 | 2,386 |

**Table 2: Top Rules by Lift**
| Rule | Support | Confidence | Lift | Count |
| :--- | :--- | :--- | :--- | :--- |
| `{Butter} -> {Banana}` | 0.001295 | 3.61% | 0.9975 | 1,295 |
| `{Feminine Hygiene Products} -> {Deodorant}` | 0.001275 | 3.54% | 0.9754 | 1,275 |
| `{BBQ Sauce} -> {Yogurt}` | 0.001267 | 3.52% | 0.9722 | 1,267 |

*Note:* All 2-item rules have `Lift < 1.0` because 20.23% of transactions only have 1 item, and the average basket is very small (2.95 items).

#### 2. Three-Item Rules (`{A, B} -> {C}`) with Positive Lift (> 1.0)
When we check customers who already bought 2 items, we know it is not a 1-item basket. We found **15 rules with Lift > 1.0**:

**Table 3: Top Rules with Positive Lift**
| Rule | Support | Confidence | Lift | Count |
| :--- | :--- | :--- | :--- | :--- |
| `{Soap, Toothpaste} -> {Dustpan}` | 0.000095 | 4.01% | **1.1242** | 95 |
| `{Canned Soup, Toothpaste} -> {Cheese}` | 0.000090 | 3.87% | **1.0769** | 90 |
| `{Apple, Toothpaste} -> {Tomatoes}` | 0.000091 | 3.84% | **1.0668** | 91 |

**Key Findings:**
1. **`{Soap, Toothpaste} -> {Dustpan}`:** Has our highest Lift (`1.1242`). Customers buying Soap and Toothpaste are **12.4% more likely** to also buy a Dustpan.
2. **`{Canned Soup, Toothpaste} -> {Cheese}`:** Has a `3.87%` confidence and `1.0769` lift (7.7% higher chance of buying Cheese).
3. **`{Tomatoes} -> {Toothpaste}`:** Has the highest confidence (`6.70%`), but its Lift is below 1 (`0.9438`) because Toothpaste is already bought in 7.09% of all transactions on its own.

   

### TASK 5 PART B

In [6]:
# TASK 5 PART B: DISCOVERING POSITIVE LIFT (LIFT > 1.0) VIA 3-ITEMSETS 
# Why 2-item rules have Lift < 1.0: Finite basket slots (avg 2.95 items) create "slot competition".
# Why 3-item rules ({A, B} -> {C}) achieve Lift > 1.0: Knowing a basket has {A, B} guarantees it is NOT a 1-item basket!

# Take the top 30 most frequent 2-item pairs and find their exact co-occurrence with a 3rd item C

top_pairs = df_2_itemsets.head(30)
rules_3_list = []

for _, row in top_pairs.iterrows():
    A, B = row['item_A'], row['item_B']
    idx_A, idx_B = product_to_idx[A], product_to_idx[B]
    supp_AB = row['support']
    
   
    mask_AB = (X[:, idx_A] == 1) & (X[:, idx_B] == 1)
    X_sub = X[mask_AB]
    
  
    counts_ABC = X_sub.sum(axis=0)
    
    for idx_C, C in enumerate(products):
        if C not in (A, B):
            count_ABC = counts_ABC[idx_C]
            supp_ABC = count_ABC / n_transactions
            supp_C = supp_dict[C]
            
            conf_AB_to_C = supp_ABC / supp_AB
            lift_AB_to_C = conf_AB_to_C / supp_C
            
            rules_3_list.append({
                'Rule': f"{{{A}, {B}}} -> {{{C}}}",
                'Support': supp_ABC,
                'Confidence': conf_AB_to_C,
                'Lift': lift_AB_to_C,
                'Count': count_ABC
            })

df_rules_3 = pd.DataFrame(rules_3_list).sort_values(by='Lift', ascending=False)

print(f"3-Item Rules with Positive Correlation (Lift > 1.0): {sum(df_rules_3['Lift'] > 1.0):,} out of {len(df_rules_3):,} tested")
print("\n--- TABLE 3: TOP 5 POSITIVE-LIFT ASSOCIATION RULES ({A, B} -> {C}) ---")
print(df_rules_3.head(5).to_string(index=False))

3-Item Rules with Positive Correlation (Lift > 1.0): 15 out of 2,370 tested

--- TABLE 3: TOP 5 POSITIVE-LIFT ASSOCIATION RULES ({A, B} -> {C}) ---
                                      Rule  Support  Confidence     Lift  Count
           {Soap, Toothpaste} -> {Dustpan} 0.000095    0.040135 1.124204     95
     {Canned Soup, Toothpaste} -> {Cheese} 0.000090    0.038743 1.076942     90
         {Apple, Toothpaste} -> {Tomatoes} 0.000091    0.038429 1.066822     91
          {Rice, Toothpaste} -> {Hair Gel} 0.000089    0.038280 1.063528     89
{Peanut Butter, Toothpaste} -> {Dish Soap} 0.000089    0.038214 1.063386     89


---

## Task 6. Discussion: Business Interpretation and Limitations

**Business Interpretation:**
* **Store Layout (Anchor Product):** `Toothpaste` is bought in **7.09%** of all transactions (almost double any other item). Stores can place everyday items or promotional products near the Toothpaste shelf since so many customers walk down that aisle.
* **Product Bundling:** Rules with positive lift like `{Soap, Toothpaste} -> {Dustpan}` (household cleaning/personal care) and `{Canned Soup, Toothpaste} -> {Cheese}` (grocery) can be used for combo discounts or "Frequently Bought Together" suggestions. Also, natural pairs like `{Butter} -> {Banana}` and `{Feminine Hygiene Products} -> {Deodorant}` can be placed in nearby aisles.
* **Increasing Basket Size:** Since 20.23% of customers buy only 1 item, the store can offer small discounts like *"Buy 3 items, get 5% off"* to encourage bigger baskets.

**Limitations:**
* **Synthetic Dataset:** This Kaggle dataset is computer-generated where items were picked mostly at random. Because of this, the Lift values are close to `1.0`, unlike real supermarket data where items like *Bread* and *Butter* have much higher Lift.
* **High Confidence Can Be Misleading:** `{Tomatoes} -> {Toothpaste}` has a 6.70% confidence only because Toothpaste is already popular on its own (7.09%), not because Tomatoes make people buy Toothpaste (Lift is 0.94). This shows why checking Lift is necessary.
* **Missing Quantity and Price Info:** Converting data to `True`/`False` ignores how many items were bought (`Total_Items`) and how much money was spent (`Total_Cost`).

---



## Task 7. Conclusion: What Was Learned from the Experiment
From this experiment on 1,000,000 retail transactions, we learned:
1. How to clean raw transaction lists and convert them into a `True`/`False` Boolean table (`1,000,000 x 81`) for data mining.
2. How changing the `min_support` threshold changes the output: a high support (`3.5%`) only finds single items (`81` items), while a lower support (`0.11%`) is needed to find item pairs (`3,112` pairs).
3. Why we must use **Support, Confidence, and Lift** together. Confidence alone can be tricked by popular items like `Toothpaste`, while Lift helps us find real positive connections (`Lift > 1.0`) in multi-item baskets.

---